# Cow Behavior Classification with Vision Transformer

Clean training/evaluation mirror of the notebook workflow.

- Trains a ViT classifier on behavior crops
- Evaluates on a held-out test split
- Saves organized visual artifacts to `artifacts/figures/vit_classifier/`



In [ ]:
import random
from collections import Counter
from pathlib import Path

import evaluate
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import DatasetDict, load_dataset
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)




In [ ]:
# Configuration
DATA_ROOT = Path("workdir/crops_raw")
MODEL_CHECKPOINT = "google/vit-base-patch16-224-in21k"
MODEL_OUTPUT_DIR = Path("artifacts/models/cow-behavior-vit")
FIGURES_DIR = Path("artifacts/figures/vit_classifier")
TRAINING_RUNS_DIR = Path("artifacts/runs/cow-behavior-vit")

EPOCHS = 10
LEARNING_RATE = 5e-5
BATCH_TRAIN = 32
BATCH_EVAL = 64
SEED = 42
SAVE_MODEL = True

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
else:
    print("Using CPU")

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Data directory not found: {DATA_ROOT}")

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRAINING_RUNS_DIR.mkdir(parents=True, exist_ok=True)




In [ ]:
# Load dataset and create stratified splits (70/15/15)
raw_dataset = load_dataset("imagefolder", data_dir=str(DATA_ROOT))["train"]
test_split = raw_dataset.train_test_split(
    test_size=0.15,
    stratify_by_column="label",
    seed=SEED,
)
train_val_split = test_split["train"].train_test_split(
    test_size=0.1765,  # 0.15 / (1 - 0.15)
    stratify_by_column="label",
    seed=SEED,
)
dataset = DatasetDict(
    train=train_val_split["train"],
    validation=train_val_split["test"],
    test=test_split["test"],
)

print(
    f"Dataset splits -> train: {len(dataset['train']):,}, "
    f"val: {len(dataset['validation']):,}, test: {len(dataset['test']):,}"
)




In [ ]:
# Class distribution summary
class_names = dataset["train"].features["label"].names
train_counts = Counter(dataset["train"]["label"])
class_distribution = pd.DataFrame(
    {
        "class": class_names,
        "count": [train_counts[i] for i in range(len(class_names))],
        "percent": [
            100.0 * train_counts[i] / len(dataset["train"])
            for i in range(len(class_names))
        ],
    }
).round(2)
print("Class distribution (train):")
print(class_distribution.to_string(index=False))




In [ ]:
# Preprocessing
processor = AutoImageProcessor.from_pretrained(MODEL_CHECKPOINT, use_fast=True)


def preprocess(examples: dict) -> dict:
    images = [img.convert("RGB") for img in examples["image"]]
    inputs = processor(images)
    return {
        "pixel_values": inputs["pixel_values"],
        "labels": examples["label"],
    }


dataset = dataset.map(preprocess, batched=True, remove_columns=["image", "label"])
for split_name in dataset:
    dataset[split_name].set_format("torch", columns=["pixel_values", "labels"])




In [ ]:
# Model and label mappings
id2label = {i: name for i, name in enumerate(class_names)}
label2id = {name: i for i, name in id2label.items()}

model = AutoModelForImageClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
).to(device)




In [ ]:
# Metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred) -> dict[str, float]:
    preds = eval_pred.predictions.argmax(-1)
    acc = accuracy_metric.compute(predictions=preds, references=eval_pred.label_ids)[
        "accuracy"
    ]
    f1_weighted = f1_metric.compute(
        predictions=preds,
        references=eval_pred.label_ids,
        average="weighted",
    )["f1"]
    return {"accuracy": acc, "f1_weighted": f1_weighted}




In [ ]:
# Training configuration
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
training_args = TrainingArguments(
    output_dir=str(TRAINING_RUNS_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_TRAIN,
    per_device_eval_batch_size=BATCH_EVAL,
    learning_rate=LEARNING_RATE,
    weight_decay=0.05,
    warmup_ratio=0.05,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    bf16=use_bf16,
    fp16=(not use_bf16 and torch.cuda.is_available()),
    report_to="none",
    save_total_limit=1,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=compute_metrics,
    processing_class=processor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)




In [ ]:
# Train
trainer.train()




In [ ]:
# Evaluate
test_metrics = trainer.evaluate(dataset["test"])
predictions = trainer.predict(dataset["test"])
y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(-1)

print("Test metrics:")
print(test_metrics)
print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=list(id2label.values())))

if SAVE_MODEL:
    trainer.save_model(str(MODEL_OUTPUT_DIR))
    processor.save_pretrained(str(MODEL_OUTPUT_DIR))
    print(f"Saved model to {MODEL_OUTPUT_DIR}")




In [ ]:
def save_confusion_matrix_figure(
    y_true_values: np.ndarray,
    y_pred_values: np.ndarray,
    labels: list[str],
    output_path: Path,
) -> None:
    cm = confusion_matrix(y_true_values, y_pred_values)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm_norm, interpolation="nearest", cmap=plt.cm.Blues)
    fig.colorbar(im, ax=ax)

    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        xticklabels=labels,
        yticklabels=labels,
        title="Confusion Matrix (Normalized)",
        ylabel="True label",
        xlabel="Predicted label",
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            value = cm_norm[i, j]
            text_color = "white" if value > 0.5 else "black"
            ax.text(j, i, f"{value:.2f}", ha="center", va="center", color=text_color)

    plt.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def save_precision_recall_figure(
    y_true_values: np.ndarray,
    y_pred_values: np.ndarray,
    labels: list[str],
    output_path: Path,
) -> None:
    precision, recall, _, _ = precision_recall_fscore_support(
        y_true_values,
        y_pred_values,
        average=None,
    )

    x = np.arange(len(labels))
    width = 0.35
    fig, ax = plt.subplots(figsize=(11, 7))
    bars_precision = ax.bar(
        x - width / 2, precision, width, label="Precision", alpha=0.85
    )
    bars_recall = ax.bar(x + width / 2, recall, width, label="Recall", alpha=0.85)

    ax.set_xlabel("Behavior")
    ax.set_ylabel("Score")
    ax.set_title("Precision and Recall by Behavior")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_ylim(0, 1.0)
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.3)

    for bar_group in [bars_precision, bars_recall]:
        for bar in bar_group:
            height = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                height,
                f"{height:.2f}",
                ha="center",
                va="bottom",
                fontsize=9,
            )

    plt.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def save_sample_predictions_grid(
    split_data,
    labels_map: dict[int, str],
    output_path: Path,
    samples_per_class: int = 2,
) -> None:
    sample_indices: list[int] = []
    by_class: dict[int, list[int]] = {i: [] for i in labels_map.keys()}

    for idx, label in enumerate(split_data["labels"]):
        label_value = int(label.item() if hasattr(label, "item") else label)
        if len(by_class[label_value]) < samples_per_class:
            by_class[label_value].append(idx)

    for class_id in sorted(by_class.keys()):
        sample_indices.extend(by_class[class_id])

    n = min(len(sample_indices), 10)
    if n == 0:
        return

    fig, axes = plt.subplots(2, 5, figsize=(16, 7))
    axes = axes.flatten()

    for i in range(10):
        axes[i].axis("off")

    for i, sample_idx in enumerate(sample_indices[:n]):
        pixel_values = split_data[sample_idx]["pixel_values"].unsqueeze(0).to(device)
        true_label = int(split_data[sample_idx]["labels"].item())

        with torch.no_grad():
            logits = model(pixel_values).logits
            pred_label = int(logits.argmax(-1).item())
            confidence = float(F.softmax(logits, dim=-1).max().item())

        image = pixel_values.squeeze().cpu().numpy().transpose(1, 2, 0)
        image = np.clip(image * 0.5 + 0.5, 0, 1)

        axes[i].imshow(image)
        axes[i].axis("off")
        color = "green" if true_label == pred_label else "red"
        axes[i].set_title(
            f"T: {labels_map[true_label]}\nP: {labels_map[pred_label]} ({confidence:.2f})",
            fontsize=9,
            color=color,
        )

    plt.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)




In [ ]:
# Save organized visual artifacts
save_confusion_matrix_figure(
    y_true_values=y_true,
    y_pred_values=y_pred,
    labels=list(id2label.values()),
    output_path=FIGURES_DIR / "confusion_matrix.png",
)

save_precision_recall_figure(
    y_true_values=y_true,
    y_pred_values=y_pred,
    labels=list(id2label.values()),
    output_path=FIGURES_DIR / "precision_recall_by_behavior.png",
)

save_sample_predictions_grid(
    split_data=dataset["test"],
    labels_map=id2label,
    output_path=FIGURES_DIR / "sample_predictions_grid.png",
)

print(f"Saved figures to {FIGURES_DIR}")
